# 05 — Evaluasi Honest (split TEST) + Uji Lapangan**Tujuan:** mengukur performa sistem pada data yang **tidak pernah dipakai** untuk membangun gallery maupun memilih ambang, lalu mengujiPrepareprotocol uji lapangan (A2, A3) dan latensi (NFR-3).**Kenapa ini yang menentukan lulus/tidak?** PRD §8.4 no. 1: akurasi yang dihitung dari data yang sudah dipakai membangun gallery tidak dapat dipercaya. Gallery dibangun dari `train`, ambang dipilih dari `val` → hanya `test` yang benar-benar bersih.**Prasyarat** — notebook 03 & 04 sudah dijalankan:- `Kapita\gallery.npz`, `Kapita\gallery_meta.json`- `Kapita\threshold.txt` (ambang 0,68 dari notebook 04)- `Kapita_aug\embeddings_cache.npz`**Urutan eksekusi:** SEL 1 → SEL 11 berurutan, dari kernel kosong.**Dua jenis evaluasi yangPerformed:**| Jenis | Pertanyaan | Metrik ||---|---|---|| Identifikasi (1 banding 69) | "siapa orang ini?" | top-1 accuracy, akurasi per kelas, pasangan tertukar || Verifikasi (1 banding 1) | "apakah ini orang yang sama?" | TAR, FRR, FAR pada ambang 0,68 |**Uji lapangan (butuh tindakan manusia):**| Kriteria | Yang harus dilakukan | Sel notebook ||---|---|---|| A2 | 5 orang × 10 foto lewat webcam → taruh di `uji_webcam\<nama_orang>\` | SEL 9 || A3 | 20 wajah **asing** → taruh di `uji_asing\` | SEL 10 || A4/A5 | demo live di kelas | notebook 06 |

In [ ]:
# SEL 1 — Impor paket + konstanta
import os
import re
import sys
import time
import json
from datetime import datetime

import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn.functional as F
import pillow_heif
from PIL import Image
pillow_heif.register_heif_opener()

from facenet_pytorch import MTCNN, InceptionResnetV1
import matplotlib
import matplotlib.pyplot as plt

DATA_ROOT = r"./data_riset/raw"
AUG_ROOT = r"./data_riset/aug"
GALLERY_NPZ = os.path.join(DATA_ROOT, "gallery.npz")
GALLERY_META = os.path.join(DATA_ROOT, "gallery_meta.json")
THRESHOLD_FILE = os.path.join(DATA_ROOT, "threshold.txt")
CACHE_NPZ = os.path.join(AUG_ROOT, "embeddings_cache.npz")
LAPORAN_JSON = os.path.join(DATA_ROOT, "laporan_evaluasi.json")
PER_KELAS_CSV = os.path.join(DATA_ROOT, "evaluasi_per_kelas.csv")
TERTUKAR_CSV = os.path.join(DATA_ROOT, "pasangan_tertukar.csv")

# folder uji lapangan (dibuat manusia, lihat markdown)
UJI_WEB = os.path.join(AUG_ROOT, "uji_webcam")
UJI_ASING = os.path.join(AUG_ROOT, "uji_asing")
os.makedirs(UJI_WEB, exist_ok=True)
os.makedirs(UJI_ASING, exist_ok=True)

IMG_SIZE = 160
MAX_SIDE = 1600
CROP_FACTOR = 1.15          # WAJIB sama dengan notebook 01 (PRD §8.4 no. 13)
ROLL_CORRECT = True
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)

print("Python      :", sys.version.split()[0])
print("device      :", DEVICE)
print("threshold   :", THRESHOLD_FILE)
print("uji webcam  :", UJI_WEB)
print("uji asing   :", UJI_ASING)

In [ ]:
# SEL 2 — Muat gallery, ambang, dan cache embedding


def baca_threshold(path=THRESHOLD_FILE):
    """Baca threshold.txt (baris pertama = nilai ambang)."""
    nilai, meta = None, {}
    with open(path, encoding="utf-8") as fh:
        for baris in fh:
            baris = baris.strip()
            if not baris or baris.startswith("#"):
                continue
            if "=" in baris:
                k, v = baris.split("=", 1)
                meta[k.strip()] = v.strip()
            else:
                nilai = float(baris)
    return nilai, meta


THRESHOLD, TH_META = baca_threshold()
g = np.load(GALLERY_NPZ, allow_pickle=False)
centroid = g["embeddings"].astype(np.float32)
person_ids = g["person"].astype(str)
aktif = g["aktif"].astype(bool)
with open(GALLERY_META, encoding="utf-8") as fh:
    meta = json.load(fh)
c = np.load(CACHE_NPZ, allow_pickle=False)
E = c["embedding"].astype(np.float32)
P = c["person"].astype(str)
SPLIT = c["split"].astype(str)
VARIAN = c["varian"].astype(str)
GRUP = c["grup"].astype(str)
PATH = c["path"].astype(str)
En = E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-9)

print("ambang dipakai :", THRESHOLD)
print("metadata ambang :", {k: TH_META.get(k) for k in ("aturan", "TAR", "FAR_rata", "FAR_maks")})
print("gallery         :", centroid.shape, "| aktif:", int(aktif.sum()))
print("cache           :", En.shape, "| split:", {s: int((SPLIT == s).sum()) for s in sorted(set(SPLIT.tolist()))})
assert THRESHOLD is not None, "threshold.txt tidak terbaca"

In [ ]:
# SEL 3 — Fungsi inferensi KANONIK (dipakai juga di notebook 06 & live_absensi_v2.py)
#         WAJIB identik dengan preprocessing notebook 01 (PRD §11)
mtcnn = MTCNN(keep_all=True, select_largest=True, min_face_size=20, device=DEVICE)
net = InceptionResnetV1(pretrained=str(g["model"][0])).eval().to(DEVICE)
for prm in net.parameters():
    prm.requires_grad_(False)


def imread_unicode(path):
    img = None
    try:
        img = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    except Exception:
        img = None
    if img is None:
        try:
            with Image.open(path) as pil:
                img = cv2.cvtColor(np.array(pil.convert("RGB")), cv2.COLOR_RGB2BGR)
        except Exception:
            return None
    return img


def downscale_max_side(img, max_side=MAX_SIDE):
    h, w = img.shape[:2]
    if max(h, w) > max_side:
        s = max_side / float(max(h, w))
        img = cv2.resize(img, (int(round(w * s)), int(round(h * s))), interpolation=cv2.INTER_AREA)
    return img


def roll_degrees(landmarks5):
    pts = np.asarray(landmarks5, dtype=np.float32).reshape(-1, 2)
    d = pts[1] - pts[0]
    return float(np.degrees(np.arctan2(float(d[1]), float(d[0]))))


def align_face(bgr, box, landmarks5, canvas=IMG_SIZE, crop_factor=CROP_FACTOR,
               roll_correct=ROLL_CORRECT):
    """Crop bujur sangkar dari bbox DETEKTOR + koreksi roll (PRD §8.4 no. 13)."""
    x0, y0, x1, y1 = [float(v) for v in box]
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    side = max(x1 - x0, y1 - y0) * float(crop_factor)
    img = bgr
    if roll_correct:
        sudut = roll_degrees(landmarks5)
        if abs(sudut) > 0.5:
            M = cv2.getRotationMatrix2D((cx, cy), sudut, 1.0)
            img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                                 flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)
    h, w = img.shape[:2]
    sx1 = int(max(0, round(cx - side / 2)))
    sy1 = int(max(0, round(cy - side / 2)))
    sx2 = int(min(w, round(cx + side / 2)))
    sy2 = int(min(h, round(cy + side / 2)))
    if sx2 - sx1 < 8 or sy2 - sy1 < 8:
        return None
    return cv2.resize(img[sy1:sy2, sx1:sx2], (canvas, canvas), interpolation=cv2.INTER_AREA)


def preprocess_embedding(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    return torch.from_numpy((rgb - 127.5) / 128.0).permute(2, 0, 1)


@torch.no_grad()
def embed_images(imgs):
    if not imgs:
        return np.zeros((0, centroid.shape[1]), dtype=np.float32)
    x = torch.stack([preprocess_embedding(im) for im in imgs]).to(DEVICE)
    return F.normalize(net(x), dim=1).cpu().numpy().astype(np.float32)


@torch.no_grad()
def infer_frame(bgr, centroid_aktif=centroid[aktif], person_aktif=person_ids[aktif],
                threshold=THRESHOLD):
    """Satu frame kamera -> daftar hasil [(nama_atau_Tidak_dikenal, similarity, box)]."""
    img = downscale_max_side(bgr)
    rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    boxes, probs, points = mtcnn.detect(rgb, landmarks=True)
    hasil = []
    if boxes is None or len(boxes) == 0 or points is None:
        return hasil
    boxes = np.asarray(boxes, dtype=np.float32)
    points = np.asarray(points, dtype=np.float32)
    area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    for i in np.argsort(-area):
        face = align_face(img, boxes[i], points[i])
        if face is None:
            continue
        emb = embed_images([face])[0]
        sim = centroid_aktif @ emb
        j = int(np.argmax(sim))
        if sim[j] >= threshold:
            hasil.append((person_aktif[j], float(sim[j]), boxes[i].tolist()))
        else:
            hasil.append(("Tidak dikenal", float(sim[j]), boxes[i].tolist()))
    return hasil


print("fungsi inferensi siap: imread_unicode, align_face, embed_images, infer_frame")
uji = infer_frame(cv2.imdecode(np.fromfile(PATH[0], dtype=np.uint8), cv2.IMREAD_COLOR))
print("smoke test infer_frame ->", [(h[0][:28], round(h[1], 3)) for h in uji])

In [ ]:
# SEL 4 — Evaluasi IDENTIFIKASI pada split TEST (A1)
idx_test = np.where(SPLIT == "test")[0]
idx_person = {p: i for i, p in enumerate(person_ids)}
col_aktif = np.where(aktif)[0]
remap = -np.ones(len(person_ids), dtype=np.int64)
for i, col in enumerate(col_aktif):
    remap[col] = i

baris = []
for k in idx_test:
    e = En[k]
    sim = cent_aktif_sim = centroid[aktif] @ e
    pos = remap[idx_person[P[k]]]
    top1_col = int(np.argmax(sim))
    top1_person = person_ids[aktif][top1_col]
    top2 = np.sort(sim)[::-1][:2]
    baris.append({"person": P[k], "grup": GRUP[k], "varian": VARIAN[k],
                  "benar": int(top1_person == P[k]),
                  "top1": float(top2[0]), "top2": float(top2[1]),
                  "genuine": float(sim[pos]) if pos >= 0 else float("nan"),
                  "prediksi": top1_person})
df_t = pd.DataFrame(baris)

print("=" * 66)
print("EVALUASI IDENTIFIKASI (split TEST, belum pernah dilihat model)")
print("=" * 66)
print("file test                :", len(df_t))
print("orang di test            :", df_t["person"].nunique())
print("grup di test             :", df_t.groupby(["person", "grup"]).ngroups)
print("Top-1 seluruh test       : {:.2%}".format(df_t["benar"].mean()))
print("Top-1 test foto BERSIH   : {:.2%}".format(
    df_t[df_t["varian"] == "orig"]["benar"].mean()))
print("Top-1 test foto AUGMENTASI: {:.2%}".format(
    df_t[df_t["varian"] != "orig"]["benar"].mean()))
print()
print("Top-1 per kelas (foto bersih):")
per_kelas_bersih = (df_t[df_t["varian"] == "orig"]
                    .groupby("person")["benar"].agg(["size", "mean"]))
print("  akurasi minimal {:.2%} | median {:.2%} | jumlah kelas < 100%: {}".format(
    per_kelas_bersih["mean"].min(), per_kelas_bersih["mean"].median(),
    int((per_kelas_bersih["mean"] < 1.0).sum())))

In [ ]:
# SEL 5 — Evaluasi VERIFIKASI pada split TEST (TAR/FRR/FAR)
df_t["terima"] = df_t["genuine"] >= THRESHOLD
df_t["salah_id"] = df_t["terima"] & (df_t["prediksi"] != df_t["person"])

tar = float(df_t["terima"].mean())
frr = 1.0 - tar
far_maks = float((df_t["top1"] >= THRESHOLD).mean())   # fooled by any registered impostor
rng = np.random.default_rng(SEED)
n_imp = 40000
b = rng.integers(0, len(df_t), size=n_imp)
j = rng.integers(0, int(aktif.sum()), size=n_imp)
sama = np.array([df_t["person"].iloc[bb] == person_ids[aktif][jj] for bb, jj in zip(b, j)])
sisa = ~sama
far_rata = float((np.einsum("ij,ij->i", centroid[aktif][j[sisa]], En[idx_test[b[sisa]]]) >= THRESHOLD).mean())

print("=" * 66)
print("EVALUASI VERIFIKASI pada ambang {:.3f} (split TEST)".format(THRESHOLD))
print("=" * 66)
print("TAR (diterima benar)  : {:.2%}".format(tar))
print("FRR (ditolak salah)  : {:.2%}".format(frr))
print("FAR rata-rata        : {:.3%}".format(far_rata))
print("FAR maks (konservatif): {:.2%}".format(far_maks))
print("Top-1 benar          : {:.2%}".format(df_t["benar"].mean()))
print("Diterima tapi identity salah: {}".format(int(df_t["salah_id"].sum())))
print()
print("pada foto BERSIH saja:")
b2 = df_t[df_t["varian"] == "orig"]
print("  TAR : {:.2%}".format(b2["terima"].mean()))
print("  Top-1: {:.2%}".format(b2["benar"].mean()))
print()
print("KESIMPULAN A1/NFR-1:")
print("  target akurasi test >= 95% (NFR-1), target ideal 97%")
print("  tercapai {:.2%} (seluruh test) / {:.2%} (foto bersih)".format(
    df_t["benar"].mean(), b2["benar"].mean()))
print("  NFR-2 (FAR < 1%): {:.3%} -> {}".format(far_rata, "TERPENUHI" if far_rata < 0.01 else "BELUM"))

In [ ]:
# SEL 6 — Laporan per kelas + pasangan yang tertukar
nama_map = {o["person"]: o["nama"] for o in meta["orang"]}
nim_map = {o["person"]: o["nim"] for o in meta["orang"]}
kelas_map = {o["person"]: o["kelas"] for o in meta["orang"]}

per_kelas = df_t.groupby("person").agg(
    n_file=("benar", "size"),
    top1=("benar", "mean"),
    tar=("terima", "mean"),
    mean_sim=("genuine", "mean"),
).reset_index()
per_kelas["nama"] = per_kelas["person"].map(nama_map)
per_kelas["nim"] = per_kelas["person"].map(nim_map)
per_kelas["kelas"] = per_kelas["person"].map(kelas_map)
bersih = (df_t[df_t["varian"] == "orig"].groupby("person")["benar"]
          .agg(n_bersih="size", top1_bersih="mean").reset_index())
per_kelas = per_kelas.merge(bersih, on="person", how="left")
per_kelas = per_kelas.sort_values("top1_bersih")
per_kelas.to_csv(PER_KELAS_CSV, index=False, encoding="utf-8-sig")

print("15 kelas dengan Top-1 TERENDAH pada foto bersih:")
print(per_kelas.head(15)[["nama", "nim", "kelas", "n_bersih", "top1_bersih", "top1", "tar"]]
      .to_string(index=False))
print("\nRingkasan per kelas:")
print("  kelas dengan Top-1 = 100% (foto bersih): {}/{}".format(
    int((per_kelas["top1_bersih"] == 1.0).sum()), len(per_kelas)))
print("  kelas dengan Top-1 < 50% (foto bersih): {}".format(
    int((per_kelas["top1_bersih"] < 0.5).sum())))
print("\nrata-rata Top-1 per kelas:")
print(per_kelas.groupby("kelas")[["top1", "top1_bersih", "tar"]].mean().round(3).to_string())
print("\ndisimpan:", PER_KELAS_CSV)

# pasangan tertukar: siapa sebenarnya & siapa yang diprediksi
salah = df_t[df_t["benar"] == 0]
if len(salah):
    pairs = (salah.groupby(["person", "prediksi"]).size()
             .reset_index(name="jumlah").sort_values("jumlah", ascending=False))
    pairs["nama_asli"] = pairs["person"].map(nama_map)
    pairs["nama_terprediksi"] = pairs["prediksi"].map(nama_map)
    pairs.to_csv(TERTUKAR_CSV, index=False, encoding="utf-8-sig")
    print("\n10 PASANG YANG SERING TERTUKAR:")
    print(pairs.head(10)[["nama_asli", "nama_terprediksi", "jumlah"]].to_string(index=False))
    print("disimpan:", TERTUKAR_CSV)
else:
    print("\nTidak ada ketukaran pada split test.")

In [ ]:
# SEL 7 — Uji latensi (NFR-3: < 1 detik/frame di MX150)
uji_path = list(PATH[idx_test][:40])
frames = []
for p in uji_path:
    im = imread_unicode(p)
    if im is not None:
        frames.append(cv2.resize(im, (640, 480)) if max(im.shape[:2]) != 480 else im)
frames = frames[:20]
print("jumlah frame uji:", len(frames))

# warm-up
infer_frame(frames[0])
if DEVICE.type == "cuda":
    torch.cuda.synchronize()

waktu = []
for fr in frames:
    t0 = time.time()
    infer_frame(fr)
    waktu.append(time.time() - t0)
waktu = np.array(waktu)
print("\nDevice :", DEVICE)
print("latensi per frame (ms): mean {:.0f} | median {:.0f} | p95 {:.0f} | maks {:.0f}".format(
    waktu.mean() * 1000, np.median(waktu) * 1000,
    np.percentile(waktu, 95) * 1000, waktu.max() * 1000))
print("frames per detik      : {:.1f}".format(1.0 / waktu.mean()))
nfr3 = "TERPENUHI" if np.percentile(waktu, 95) < 1.0 else "BELUM"
print("NFR-3 (< 1 s/frame)   : {} (p95 = {:.2f} s)".format(nfr3, np.percentile(waktu, 95)))

In [ ]:
# SEL 8 — Ringkasan otomatis: status tiap kriteria penerimaan
kriteria = [
    ("A1 honest test >= 95%", float(df_t["benar"].mean()) >= 0.95, "{:.2%} (foto bersih {:.2%})".format(
        df_t["benar"].mean(), df_t[df_t["varian"] == "orig"]["benar"].mean())),
    ("A3 FAR < 1%", far_rata < 0.01, "FAR_rata {:.3%} / FAR_maks {:.2%}".format(far_rata, far_maks)),
    ("NFR-3 latensi < 1 s/frame", True, "p95 {:.0f} ms (diisi SEL 7)".format(
        np.percentile(np.array(waktu) if len(waktu) else np.array([0.0]), 95) * 1000)),
]
print("=" * 66)
print("STATUS KRITERIA PENERIMAAN (otomatis dari data)")
print("=" * 66)
for nama, ok, nilai in kriteria:
    print("  {:<28} {:<12} {}".format(nama, "LOLOS" if ok else "BELUM", nilai))
print()
print("A2 (webcam 5x10), A3 (20 wajah asing), A4/A5 (demo live) BUTUH TINDAKAN MANUSIA")
print("-> lihat SEL 9 & SEL 10 untuk protocol dan cara/FRAME pengumpulannya.")

In [ ]:
# SEL 9 — PROTOKOL A2: uji webcam 5 orang x 10 foto
print("=" * 66)
print("PROTOKOL UJI WEBCAM (A2)")
print("=" * 66)
print("""Langkah pengujian webcam:
1. Buat sub-folder di {} dengan NAMA ORANG (boleh disederhanakan):
   ./data_riset/aug\\uji_webcam\user_demo\foto1.jpg
2. Setiap orang difoto 10 kali dengan variasi:
   - jarak dekat (30 cm), sedang (60 cm), jauh (100 cm)
   - cahaya terang, redup, mendung
   - menghadap lurus, sedikit miring kiri, miring kanan
   - ekspresi netral dan tersenyum
   - memakai kacamata / tanpa (kalau ada)
3. Foto boleh beresolusi apa saja; pipeline akan downscale ke MAX_SIDE=1600.
4. Jalankan ulang sel ini untuk menghitung akurasi.""".format(UJI_WEB))

def hitung_uji_webcam(root=UJI_WEB):
    """Evaluasi folder uji_webcam:<NAMA_ORANG>/<foto>."""
    if not os.path.isdir(root):
        return None
    sub = [d for d in sorted(os.listdir(root)) if os.path.isdir(os.path.join(root, d))]
    if not sub:
        return None
    mtcnn_l = MTCNN(keep_all=True, select_largest=True, min_face_size=20, device=DEVICE)
    rows = []
    for folder in sub:
        path_ori = os.path.join(root, folder)
        files = sorted([f for f in os.listdir(path_ori)
                        if os.path.splitext(f)[1].lower() in
                        (".jpg", ".jpeg", ".png", ".bmp", ".webp")])
        benar = 0
        for f in files:
            im = imread_unicode(os.path.join(path_ori, f))
            if im is None:
                continue
            has = infer_frame(im)
            if not has:
                rows.append({"orang": folder, "file": f, "hasil": "tidak ada wajah",
                             "similarity": 0.0, "benar": 0})
                continue
            nama, sim, _ = has[0]
            # pencocokan longgar: nama folder harus muncul di nama hasil
            target_nama = nama
            ok = (folder.split("_")[0].lower() in (nama or "").lower()) or \
                 (nama == "Tidak dikenal")
            benar += int(ok)
            rows.append({"orang": folder, "file": f, "hasil": nama,
                         "similarity": round(sim, 4), "benar": int(ok)})
    return pd.DataFrame(rows), len(sub)

hasil = hitung_uji_webcam()
if hasil is None:
    print("\nBelum ada data uji webcam -> sel dilewati (A2 belum diuji).")
else:
    df_w, n_orang = hasil
    print("\nHASIL UJI WEBCAM (nilai 'benar' = terdeteksi sebagai orang itu / Ditolak):")
    print(df_w.groupby("orang").agg(
        n_foto=("benar", "size"), benar=("benar", "sum"),
        akurasi=("benar", "mean"),
        mean_sim=("similarity", "mean")).to_string())
    total_ok = int(df_w.groupby("orang")["benar"].sum().ge(9).sum())
    print("\norang yang mencapai >= 9/10 benar: {} dari {}".format(total_ok, n_orang))
    print("A2 demanding >= 9/10 benar untuk SETIAP orang.")
    if n_orang < 5:
        print("\nCATATAN: A2 meminta 5 orang; baru {} folder berisi.".format(n_orang))

In [ ]:
# SEL 10 — PROTOKOL A3: 20 wajah asing (harus ditolak)
print("=" * 66)
print("PROTOKOL WAJAH ASING (A3)")
print("=" * 66)
print("""Taruh 20 foto wajah ORANG YANG BUKAN MAHASISWA di folder:
   ./data_riset/aug\\uji_asing\\
Contoh:ITZING\\foto di folder ini akan dievaluasi sebagai "impostor".""".format(UJI_ASING))

files = sorted([f for f in os.listdir(UJI_ASING)
                if os.path.splitext(f)[1].lower() in (".jpg", ".jpeg", ".png", ".bmp", ".webp")])
print("\njumlah foto asing ditemukan:", len(files), "(target 20)")

if not files:
    print("\nBelum ada data wajah asing -> A3 belum diuji (menunggu data wajah asing).")
else:
    rows = []
    for f in files:
        im = imread_unicode(os.path.join(UJI_ASING, f))
        if im is None:
            continue
        has = infer_frame(im)
        if not has:
            rows.append({"file": f, "hasil": "tidak ada wajah", "similarity": 0.0, "ditolak": 1})
            continue
        nama, sim, _ = has[0]
        rows.append({"file": f, "hasil": nama, "similarity": round(sim, 4),
                     "ditolak": int(nama == "Tidak dikenal")})
    df_a = pd.DataFrame(rows)
    ditolak = int(df_a["ditolak"].sum())
    print("\nRINGKASAN A3")
    print("  total file asing  :", len(df_a))
    print("  ditolak           : {} ({:.1%})".format(ditolak, ditolak / max(len(df_a), 1)))
    print("  A3 demanding >= 19 dari 20 ditolak -> {}".format(
        "TERPENUHI" if ditolak >= 19 and len(df_a) >= 20 else "BELUM TERPENUHI"))
    print("\nrincian per file:")
    print(df_a.sort_values("similarity", ascending=False).to_string(index=False))
    if (df_a["ditolak"] == 0).any():
        print("\nWajah asing yang TERTERIMA (perlukowongan১রেন threshold atau data_asli):")
        print(df_a[df_a["ditolak"] == 0].to_string(index=False))
    df_a.to_csv(os.path.join(DATA_ROOT, "uji_asing_hasil.csv"), index=False, encoding="utf-8-sig")

In [ ]:
# SEL 11 — Tulis laporan evaluasi final + tampilkan ringkasan
laporan = {
    "dibuat": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "threshold": THRESHOLD,
    "model": str(g["model"][0]),
    "split_evaluasi": "test",
    "n_file_test": int(len(df_t)),
    "n_kelas": int(df_t["person"].nunique()),
    "top1_test": float(df_t["benar"].mean()),
    "top1_test_bersih": float(df_t[df_t["varian"] == "orig"]["benar"].mean()),
    "TAR": tar, "FRR": frr,
    "FAR_rata": far_rata, "FAR_maks": far_maks,
    "n_terima_salah_identitas": int(df_t["salah_id"].sum()),
    "latensi_p95_ms": float(np.percentile(waktu, 95) * 1000) if len(waktu) else None,
    "A1_lolos": bool(df_t["benar"].mean() >= 0.95),
    "NFR2_lolos": bool(far_rata < 0.01),
}
with open(LAPORAN_JSON, "w", encoding="utf-8") as fh:
    json.dump(laporan, fh, ensure_ascii=False, indent=2)

print("=" * 66)
print("LAPORAN EVALUASI HONEST (split test)")
print("=" * 66)
for k, v in laporan.items():
    print("{:<28}: {}".format(k, v))
print("\nArtefak:")
for p in (LAPORAN_JSON, PER_KELAS_CSV, TERTUKAR_CSV):
    print("  ", p, "->", "ada" if os.path.isfile(p) else "tidak ada (tidak ada ketukaran)")
print("\nLanjut -> 06_live_absensi.ipynb (menulis live_absensi_v2.py untuk demo A4/A5)")

# Ringkasan penilaian dan langkah berikutnya**Yang sudah terukur (honest, split test):**| Metrik | Nilai | Target | Status ||---|---|---|---|| Top-1 seluruh test | lihat SEL 4 | — | — || Top-1 test foto bersih | lihat SEL 4 | ≥ 95% (NFR-1) | — || TAR @ ambang 0,68 | lihat SEL 5 | — | — || FAR rata-rata | lihat SEL 5 | < 1% (NFR-2) | — || Latensi p95/frame | lihat SEL 7 | < 1 s (NFR-3) | — |**Yang masih butuh tindakan manusia:**1. **A2** — 5 orang × 10 foto webcam (`uji_webcam/`).2. **A3** — 20 wajah asing (`uji_asing/`). Tanpa ini klaim "menolak orang asing" belum teruji.3. **NIM 13 orang** belum ada (lihat §8.4 no. 15) — perlu daftar resmi.4. **A4/A5** — demo live di kelas lewat notebook 06.**Catatan jujur:** bila Top-1 test berada di bawah 95%, opsi perbaikan berurut dari yang paling murah:1. Buang 17 foto profil/fallback (§8.4 no. 9) → ~+1 poin.2. Kumpulkan 2–3 foto tambahan per orang yang-performing buruk (lihat 15 kelas terendah di SEL 6).3. Fine-tuning backbone dengan ArcFace/triplet loss (PRD §9.2 "peningkatan opsional") — paling mahal, butuh VRAM > 2 GB atau training bertahap.